In [ ]:
import torch
import numpy as np
import sympy as sp
import os, sys
import symbolicregression
import requests
from IPython.display import display

/home/jake/miniconda3/envs/symbolic_regression/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import numpy as np
torch.manual_seed(0)
np.random.seed(0)

In [3]:
model_path = "model.pt" 
try:
    if not os.path.isfile(model_path): 
        url = "https://dl.fbaipublicfiles.com/symbolicregression/model1.pt"
        r = requests.get(url, allow_redirects=True)
        open(model_path, 'wb').write(r.content)
    if not torch.cuda.is_available():
        model = torch.load(model_path, map_location=torch.device('cpu'))
    else:
        model = torch.load(model_path)
        model = model.cuda()
    print(model.device)
    print("Model successfully loaded!")

except Exception as e:
    print("ERROR: model not loaded! path was: {}".format(model_path))
    print(e)    

cuda:0
Model successfully loaded!


In [4]:
est = symbolicregression.model.SymbolicTransformerRegressor(
                        model=model,
                        max_input_points=200,
                        n_trees_to_refine=100,
                        rescale=True
                        )

In [5]:
##Example of data

x = np.random.randn(100, 2)
y = np.cos(2*np.pi*x[:,0])+x[:,1]**2


In [6]:
torch.manual_seed(0)
np.random.seed(0)
est.fit(x,y)
actions_taken = est.model.decoder.actions_taken
print(actions_taken)
replace_ops = {"add": "+", "mul": "*", "sub": "-", "pow": "**", "inv": "1/"}
model_str = est.retrieve_tree(with_infos=True)["relabed_predicted_tree"].infix()
for op,replace_op in replace_ops.items():
    model_str = model_str.replace(op,replace_op)
display(sp.parse_expr(model_str))

Is scaling
Generating
torch.Size([1, 100, 512]), torch.Size([1])
MODEL_WRAPPER generations
[((0.01 add (0.9900000000000001 mul cos((-5.7 mul x_0)))) add ((0.001 add (0.04 mul x_1)) mul (6.6000000000000005 add (26.0 mul x_1))))]
MODEL_WRAPPER generations
Sampling
Generating
torch.Size([10, 100, 512]), torch.Size([10])
MODEL_WRAPPER generations
[[((0.01 add (0.9900000000000001 mul cos((-5.7 mul x_0)))) add ((0.001 add (0.04 mul x_1)) mul (6.6000000000000005 add (26.0 mul x_1)))), ((0.00396 add (-0.932 mul cos((53.7 add (5.8 mul x_0))))) sub ((-0.05450000000000001 add (-35.6 mul x_1)) mul (0.008029999999999999 add (0.0296 mul x_1)))), (((3.64 add (-0.00554 mul x_1)) mul ((-0.025200000000000004 add (-0.0753 mul ((0.14 add x_1))**2)) mul (-3.09 add (5.5200000000000005 mul sin((-89.3 add (-0.00618 mul x_0))))))) sub (0.07700000000000001 add (-0.9420000000000001 mul cos(((0.09780000000000001 add (-7.99 mul x_0)) add ((-0.87 add (-0.00503 mul x_0)) mul (71.4 add (-0.317 mul arctan(((0.00392 ad

0.001220694228029737*x_1 - 0.9500000000000001*sin(5.74387655551529*x_0 + 61.40550080606491) + 0.8520000000000001*Abs(1.167251809711353*(x_1 + 0.041376687735097083)**2 - 0.007489760071586314*(x_0 + 0.4707048256905229*x_1 - 0.046111590110111392)**2 + 0.08085000000000001) - 0.052074289096019449

In [ ]:
torch.manual_seed(0)
np.random.seed(0)
state = est.init_state(x,y)
i = 0
while est.is_solution(state) == False:
    a = actions_taken[i]  # This will not crash so long as we have is_solution correct, otherwise we may 
                          #   index out of bound
    state = est.apply_action(state, idx=a)
    i = i + 1
assert(i == len(actions_taken)) # this checks that we applied all actions as the original model
assert(est.is_solution(state) == True)
replace_ops = {"add": "+", "mul": "*", "sub": "-", "pow": "**", "inv": "1/"}
est.prep_for_refinement(state)
model_str = est.retrieve_tree(with_infos=True)["relabed_predicted_tree"].infix()
for op,replace_op in replace_ops.items():
    model_str = model_str.replace(op,replace_op)
display(sp.parse_expr(model_str))

Is scaling
MIDDLE init_state
INNER init_state
DEBUG FIGURE OUT WHAT HAPPENS HERE
Sampling
Generating
torch.Size([10, 100, 512]), torch.Size([10])
MODEL_WRAPPER sample_candidates_from_state
[[((0.01 add (0.9900000000000001 mul cos((-5.7 mul x_0)))) add ((0.001 add (0.04 mul x_1)) mul (6.6000000000000005 add (26.0 mul x_1)))), ((0.00396 add (-0.932 mul cos((53.7 add (5.8 mul x_0))))) sub ((-0.05450000000000001 add (-35.6 mul x_1)) mul (0.008029999999999999 add (0.0296 mul x_1)))), (((3.64 add (-0.00554 mul x_1)) mul ((-0.025200000000000004 add (-0.0753 mul ((0.14 add x_1))**2)) mul (-3.09 add (5.5200000000000005 mul sin((-89.3 add (-0.00618 mul x_0))))))) sub (0.07700000000000001 add (-0.9420000000000001 mul cos(((0.09780000000000001 add (-7.99 mul x_0)) add ((-0.87 add (-0.00503 mul x_0)) mul (71.4 add (-0.317 mul arctan(((0.00392 add (0.1 mul x_0)) add (0.454 add (0.09670000000000001 mul sqrt((97.5 add (9.05 mul x_0))))))))))))))), (((-4.57 mul x_1) mul ((((0.667 mul x_0) add (31.0 add

0.001220694228029737*x_1 - 0.9500000000000001*sin(5.74387655551529*x_0 + 61.40550080606491) + 0.8520000000000001*Abs(1.167251809711353*(x_1 + 0.041376687735097083)**2 - 0.007489760071586314*(x_0 + 0.4707048256905229*x_1 - 0.046111590110111392)**2 + 0.08085000000000001) - 0.052074289096019449